# 8. Predictive Analysis — Classification

Compare Logistic Regression, K-Nearest Neighbors, Support Vector Machine, and Decision Tree classifiers for Falcon 9 landing success.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
df=pd.read_csv('../data/clean_spacex.csv')
X=pd.get_dummies(df[['PayloadMass','Flights','GridFins','Reused','Legs','Block','ReusedCount','Orbit','LaunchSite']],columns=['Orbit','LaunchSite'],dtype=int)
y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
models={
'Logistic Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000))]),
'KNN':Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=5))]),
'SVM':Pipeline([('scale',StandardScaler()),('model',SVC(kernel='rbf',C=1,probability=True))]),
'Decision Tree':DecisionTreeClassifier(max_depth=4,random_state=42)
}
results=[]
for name,model in models.items():
    model.fit(X_train,y_train); pred=model.predict(X_test)
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred)})
results=pd.DataFrame(results).sort_values('Accuracy',ascending=False)
results

,Model,Accuracy
0,Logistic Regression,0.777778
1,KNN,0.777778
2,SVM,0.777778
3,Decision Tree,0.777778


In [2]:
best_name=results.iloc[0]['Model']; best=models[best_name]
pred=best.predict(X_test)
print('Best model:',best_name)
print(classification_report(y_test,pred,digits=3))
print('Confusion matrix:')
print(confusion_matrix(y_test,pred))

Best model: Logistic Regression
              precision    recall  f1-score   support

           0      1.000     0.333     0.500         6
           1      0.750     1.000     0.857        12

    accuracy                          0.778        18
   macro avg      0.875     0.667     0.679        18
weighted avg      0.833     0.778     0.738        18

Confusion matrix:
[[ 2  4]
 [ 0 12]]


In [3]:
from sklearn.inspection import permutation_importance
imp=permutation_importance(best,X_test,y_test,n_repeats=20,random_state=42)
importance=pd.Series(imp.importances_mean,index=X.columns).sort_values(ascending=False).head(10)
importance

ReusedCount    0.066667
Legs           0.016667
Orbit_ISS      0.008333
Reused         0.000000
Orbit_ES-L1    0.000000
Orbit_MEO      0.000000
Orbit_SO       0.000000
Orbit_SSO      0.000000
Block          0.000000
Orbit_GEO      0.000000
dtype: float64

In [4]:
print('Interpretation: model performance is driven by mission profile, launch site and booster-history variables.')
print('The small sample size means results should be treated as directional rather than a production-grade forecast.')
results.to_csv('../data/model_results.csv',index=False)

Interpretation: model performance is driven by mission profile, launch site and booster-history variables.
The small sample size means results should be treated as directional rather than a production-grade forecast.
